# 🏗️ Land Development Services — AI Customer Support

Bilingual (English / Spanish) Q&A assistant for:
- 🏢 Commercial permits
- 🏠 Residential permits
- 📐 Plot / Land verification before permanent construction

**Features:**
- Detects question language and responds in the same language
- Tracks response turnaround time
- Asks for satisfaction feedback after every answer
- Saves full session log to `session_log.json`
- Prints a session summary report at the end

---

In [ ]:
# --- CELL 1: Imports ---
# dotenv     → loads OPENAI_API_KEY from .env file
# OpenAI     → official Python client for the OpenAI API
# time       → used to measure response turnaround time
# datetime   → timestamps for the session log
# json       → saves the session log to a .json file
# pathlib    → reads the FAQ file cleanly
# IPython    → renders Markdown in the notebook output

import time
import json
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

print("✅ Libraries loaded")

In [ ]:
# --- CELL 2: Configuration ---

MODEL = 'gpt-4o-mini'       # fast and cost-efficient for customer support
LOG_FILE = 'session_log.json'  # all Q&A sessions are saved here

print(f"✅ Model: {MODEL}")

In [ ]:
# --- CELL 3: Environment Setup ---
# load_dotenv() reads the .env file and makes OPENAI_API_KEY available
# to the OpenAI() client automatically — no need to pass it manually.

load_dotenv()
openai = OpenAI()
print("✅ Connected to OpenAI")

In [ ]:
# --- CELL 4: Load FAQ as AI Context ---
# We read the faq.md file and inject it into the system prompt.
# This gives the AI real department knowledge to answer from.
# When you get the real FAQ from the department, just replace faq.md.

faq_content = Path('faq.md').read_text(encoding='utf-8')

# The system prompt defines the AI's role and behavior.
# It instructs the AI to:
#   1. Act as a helpful municipal department assistant
#   2. Detect the language of the question and respond in the same language
#   3. Use the FAQ as its primary knowledge source
#   4. Be concise, professional, and helpful

SYSTEM_PROMPT = f"""You are a bilingual customer support assistant for the Land Development Services department.
You help citizens with questions about commercial permits, residential permits, and plot/land verification 
before permanent construction.

IMPORTANT RULES:
- Detect the language of the citizen's question (English or Spanish)
- Always respond in the SAME language as the question
- Be professional, clear, and concise
- Base your answers on the FAQ below when possible
- If the question is not covered in the FAQ, give your best helpful answer and suggest contacting the office
- Never make up permit fees, timelines, or legal requirements not in the FAQ

--- DEPARTMENT FAQ ---
{faq_content}
--- END FAQ ---
"""

print("✅ FAQ loaded into AI context")
print(f"   FAQ length: {len(faq_content)} characters")

In [ ]:
# --- CELL 5: Session Log Setup ---
# The session log stores every Q&A interaction.
# Each entry includes: question, response, language, response time, satisfaction.
# At the end of the session it is saved to session_log.json.

session_log = []
session_start = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print(f"✅ Session started: {session_start}")

In [ ]:
# --- CELL 6: Core Functions ---

def ask_question(question: str) -> dict:
    """
    Sends a citizen's question to the AI and returns the full interaction record.
    
    Steps:
    1. Records the timestamp when the question was asked
    2. Sends the question to GPT-4o-mini with streaming (shows answer word by word)
    3. Measures the total response time
    4. Returns a dict with all interaction data for the session log
    """
    print(f"\n{'='*60}")
    print(f"📋 Question received at: {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'='*60}\n")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question}
    ]

    # Start the timer — we measure from when the API call is made
    start_time = time.time()

    # stream=True → shows the answer token by token as it's generated
    # This feels faster to the citizen even if total time is the same
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    # Stream the response live into the notebook output
    response_text = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        response_text += chunk.choices[0].delta.content or ""
        update_display(
            Markdown(response_text.replace("```", "").replace("markdown", "")),
            display_id=display_handle.display_id
        )

    # Stop the timer once the full response has been received
    response_time = round(time.time() - start_time, 2)

    print(f"\n⏱️  Response time: {response_time} seconds")

    return {
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        "question": question,
        "response": response_text,
        "response_time_seconds": response_time,
        "satisfied": None,        # filled in by get_satisfaction()
        "needs_human": None       # filled in by get_satisfaction()
    }


def get_satisfaction(record: dict) -> dict:
    """
    Asks the citizen two follow-up questions after receiving an answer:
    1. Was the answer helpful? (yes/no)
    2. If not helpful — would they like to speak to a human agent?
    
    Updates and returns the interaction record with satisfaction data.
    """
    print("\n" + "-"*60)
    satisfied = input("\n💬 Was this answer helpful? (yes / no): ").strip().lower()

    # Normalize input — accept 'y', 'yes', 'si', 'sí' as positive
    is_satisfied = satisfied in ['yes', 'y', 'si', 'sí', 'si']
    record["satisfied"] = is_satisfied

    if is_satisfied:
        print("\n✅ Thank you! / ¡Gracias! We're glad we could help.")
        record["needs_human"] = False
    else:
        # If not satisfied, offer to escalate to a human agent
        escalate = input("\n🙋 Would you like to speak to a human agent? (yes / no): ").strip().lower()
        needs_human = escalate in ['yes', 'y', 'si', 'sí']
        record["needs_human"] = needs_human

        if needs_human:
            print("\n📞 Please call: (555) 000-1234  |  Mon–Fri 8AM–5PM")
            print("📧 Or email: permits@landdevelopment.gov")
        else:
            print("\nThank you for using our service. / Gracias por usar nuestro servicio.")

    return record


def save_session_log():
    """
    Saves the full session log to session_log.json.
    Each run appends to the existing file so history is preserved.
    """
    log_path = Path(LOG_FILE)

    # Load existing log if file exists, otherwise start fresh
    existing = []
    if log_path.exists():
        with open(log_path, 'r') as f:
            existing = json.load(f)

    existing.extend(session_log)

    with open(log_path, 'w') as f:
        json.dump(existing, f, indent=2, ensure_ascii=False)

    print(f"\n💾 Session log saved to {LOG_FILE}")


def session_summary():
    """
    Prints a summary report of the current session:
    - Total questions asked
    - Average response time
    - Satisfaction rate
    - How many citizens needed a human agent
    """
    if not session_log:
        print("No questions asked in this session.")
        return

    total          = len(session_log)
    avg_time       = round(sum(r['response_time_seconds'] for r in session_log) / total, 2)
    satisfied      = sum(1 for r in session_log if r.get('satisfied') is True)
    needs_human    = sum(1 for r in session_log if r.get('needs_human') is True)
    satisfaction_pct = round((satisfied / total) * 100)

    summary = f"""
## 📊 Session Summary Report
**Session started:** {session_start}  
**Session ended:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

| Metric | Value |
|--------|-------|
| Total questions | {total} |
| Avg response time | {avg_time}s |
| Satisfied citizens | {satisfied}/{total} ({satisfaction_pct}%) |
| Needed human agent | {needs_human} |
"""
    display(Markdown(summary))


print("✅ Functions ready")

In [ ]:
# --- CELL 7: Ask a Question ---
# Change the question below to anything a citizen might ask.
# The AI will detect the language and respond in kind.
#
# English examples:
#   "What permits do I need to build a new home?"
#   "How do I verify if my land is approved for construction?"
#   "How long does a commercial permit take?"
#
# Spanish examples:
#   "¿Qué permisos necesito para construir una casa?"
#   "¿Cómo verifico si mi terreno está aprobado para construcción permanente?"
#   "¿Cuánto tiempo tarda un permiso comercial?"

question = "What permits do I need to build a new home?"

# Ask the question and get the response
record = ask_question(question)

# Ask for satisfaction feedback
record = get_satisfaction(record)

# Add to session log
session_log.append(record)

In [ ]:
# --- CELL 8: Ask Another Question (Spanish) ---
# Run this cell to ask a second question.
# Duplicate this cell as many times as needed during a support session.

question = "¿Cómo verifico si mi terreno está aprobado para construcción permanente?"

record = ask_question(question)
record = get_satisfaction(record)
session_log.append(record)

In [ ]:
# --- CELL 9: Interactive Mode ---
# Uncomment the lines below to run a live multi-question session.
# Type your question when prompted. Type 'quit' or 'salir' to end the session.

# print("Land Development Services — Customer Support")
# print("Type your question in English or Spanish. Type 'quit' to end.\n")
#
# while True:
#     q = input("Your question / Su pregunta: ").strip()
#     if q.lower() in ['quit', 'exit', 'salir']:
#         print("Session ended. / Sesión finalizada.")
#         break
#     if q:
#         record = ask_question(q)
#         record = get_satisfaction(record)
#         session_log.append(record)

In [ ]:
# --- CELL 10: Save Log & Print Summary ---
# Run this cell at the end of every support session.
# It saves the log to session_log.json and prints the summary report.

save_session_log()
session_summary()